In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.ao.quantization import QuantStub, DeQuantStub

# --- 1. DEFINIÇÃO DO MODELO PARAMETRIZADO ---
class LeNet5_Quantized(nn.Module):
    def __init__(self, activation_type="relu"):
        super(LeNet5_Quantized, self).__init__()
        
        # Stubs para converter float para int8 e vice-versa no hardware
        self.quant = QuantStub()
        self.dequant = DeQuantStub()
        
        # Seleção de ativação (útil para testar o que melhor se adapta ao hardware)
        if activation_type == "hardtanh":
            self.activation = nn.Hardtanh(min_val=-1.0, max_val=1.0)
        elif activation_type == "leaky_relu":
            self.activation = nn.LeakyReLU(0.1)
        else:
            self.activation = nn.ReLU()

        # Camadas da LeNet-5
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # Início da zona de 8 bits
        x = self.quant(x)
        
        x = self.pool1(self.activation(self.conv1(x)))
        x = self.pool2(self.activation(self.conv2(x)))
        
        x = x.reshape(-1, 16 * 5 * 5)
        
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)
        
        # Fim da zona de 8 bits
        x = self.dequant(x)
        return x

# --- 2. CONFIGURAÇÃO E DADOS ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# MNIST precisa de 32x32 para a LeNet original
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)

# Criando a instância (escolha aqui: "relu", "hardtanh" ou "leaky_relu")
model = LeNet5_Quantized(activation_type="relu").to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- 3. LOOP DE TREINAMENTO ---
print(f"Iniciando treino no {device} com Signed 8-bit ready architecture...")

for epoch in range(10): # épocas para teste rápido
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if i % 200 == 199:
            print(f'[Época {epoch + 1}, Lote {i + 1}] Loss: {loss.item():.3f}')

print("Treinamento concluído!")

# Salvar o modelo (Weights em Float32 antes da conversão final para Int8)
torch.save(model.state_dict(), "lenet5_float.pth")

Iniciando treino no cpu com Signed 8-bit ready architecture...
[Época 1, Lote 200] Loss: 0.262
[Época 1, Lote 400] Loss: 0.122
[Época 1, Lote 600] Loss: 0.199
[Época 1, Lote 800] Loss: 0.061
[Época 2, Lote 200] Loss: 0.111
[Época 2, Lote 400] Loss: 0.013
[Época 2, Lote 600] Loss: 0.011
[Época 2, Lote 800] Loss: 0.068
[Época 3, Lote 200] Loss: 0.069
[Época 3, Lote 400] Loss: 0.021
[Época 3, Lote 600] Loss: 0.020
[Época 3, Lote 800] Loss: 0.005
[Época 4, Lote 200] Loss: 0.043
[Época 4, Lote 400] Loss: 0.066
[Época 4, Lote 600] Loss: 0.012
[Época 4, Lote 800] Loss: 0.011
[Época 5, Lote 200] Loss: 0.008
[Época 5, Lote 400] Loss: 0.002
[Época 5, Lote 600] Loss: 0.034
[Época 5, Lote 800] Loss: 0.009
[Época 6, Lote 200] Loss: 0.008
[Época 6, Lote 400] Loss: 0.003
[Época 6, Lote 600] Loss: 0.016
[Época 6, Lote 800] Loss: 0.006
[Época 7, Lote 200] Loss: 0.038
[Época 7, Lote 400] Loss: 0.002
[Época 7, Lote 600] Loss: 0.025
[Época 7, Lote 800] Loss: 0.010
[Época 8, Lote 200] Loss: 0.010
[Época 8,

In [7]:
import torch.ao.quantization as quant

# 1. Preparação: Coloca o modelo em modo de avaliação e CPU
model.eval()
model.to('cpu')

# 2. Configuração: Define como queremos quantizar (Simétrica, 8-bit)
# 'fbgemm' é o motor para PCs x86 comuns
model.qconfig = quant.get_default_qconfig('fbgemm')

# 3. Inserção de Observadores: O PyTorch coloca "espiões" em cada camada
model_prepared = quant.prepare(model)

# 4. CALIBRAÇÃO (Passo Crítico):
# Vamos rodar algumas imagens pelo modelo para os "espiões" verem os ranges
print("Calibrando escalas com dados reais...")
with torch.no_grad():
    for i, (inputs, _) in enumerate(train_loader):
        model_prepared(inputs)
        if i > 50: break # 50 lotes são suficientes para calibrar

# 5. CONVERSÃO: O momento da verdade. O modelo vira INT8.
model_int8 = quant.convert(model_prepared)

print("✓ Modelo convertido para INT8 com sucesso!")

# TESTE RÁPIDO: Veja se o primeiro peso da Conv1 agora é um inteiro
p_quant = model_int8.conv1.weight().int_repr()[0][0][0][0]
print(f"Exemplo de peso quantizado na Conv1: {p_quant.item()}")

C:\Users\Norian\AppData\Local\Temp\ipykernel_26164\2288884068.py:12: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = quant.prepare(model)
c:\Users\Norian\miniconda3\envs\ia_env\lib\site-packages\torch\ao\quantization\observer.py:246: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be

Calibrando escalas com dados reais...
✓ Modelo convertido para INT8 com sucesso!
Exemplo de peso quantizado na Conv1: -66


C:\Users\Norian\AppData\Local\Temp\ipykernel_26164\2288884068.py:23: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = quant.convert(model_prepared)


In [8]:
# 1. Salva o modelo quantizado completo (Estado + Estrutura)
# Nota: O state_dict de um modelo quantizado é mais complexo que o float
torch.save(model_int8.state_dict(), "lenet5_quantized.pth")

print("✓ Parâmetros quantizados (Weights, Biases e Scales) salvos em 'lenet5_quantized.pth'")

# 2. Verificação manual (O que você vai escrever no texto do TCC)
# Vamos ver a escala da Conv1. É daqui que vem o seu SHIFT!
conv1_scale = model_int8.conv1.scale
conv1_shift = int(np.round(-np.log2(conv1_scale)))

print(f"Relatório para o VHDL:")
print(f"Camada Conv1 - Escala: {conv1_scale:.6f} -> SHIFT Sugerido: {conv1_shift}")

✓ Parâmetros quantizados (Weights, Biases e Scales) salvos em 'lenet5_quantized.pth'
Relatório para o VHDL:
Camada Conv1 - Escala: 0.069699 -> SHIFT Sugerido: 4


In [3]:
import torch
import torch.nn as nn
import torch.ao.quantization as quant
from torch.ao.quantization import QuantStub, DeQuantStub
import os
import numpy as np

# =====================================================================
# 1. A "PLANTA" DA CASA (O PyTorch precisa disso para saber a estrutura)
# =====================================================================
class LeNet5_Quantized(nn.Module):
    def __init__(self, activation_type="relu"):
        super(LeNet5_Quantized, self).__init__()
        self.quant = QuantStub()
        self.dequant = DeQuantStub()
        
        if activation_type == "hardtanh":
            self.activation = nn.Hardtanh(min_val=-1.0, max_val=1.0)
        elif activation_type == "leaky_relu":
            self.activation = nn.LeakyReLU(0.1)
        else:
            self.activation = nn.ReLU()

        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.quant(x)
        x = self.pool1(self.activation(self.conv1(x)))
        x = self.pool2(self.activation(self.conv2(x)))
        x = x.reshape(-1, 16 * 5 * 5)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)
        x = self.dequant(x)
        return x

# =====================================================================
# 2. CONSTRUINDO A CASA E PREPARANDO PARA RECEBER OS MÓVEIS INT8
# =====================================================================
model = LeNet5_Quantized(activation_type="relu")
model.eval()

# Avisa o PyTorch que essa carcaça vai trabalhar com INT8
model.qconfig = quant.get_default_qconfig('fbgemm')
model_prepared = quant.prepare(model)
model_int8 = quant.convert(model_prepared)

# =====================================================================
# 3. COLOCANDO OS MÓVEIS (Lendo o arquivo que você já tem salvo)
# =====================================================================
arquivo_quantizado = "lenet5_quantized.pth"

print("==================================================")
if os.path.exists(arquivo_quantizado):
    print(f"Carregando pesos e escalas de '{arquivo_quantizado}'...")
    model_int8.load_state_dict(torch.load(arquivo_quantizado))
    print("✓ Modelo INT8 restaurado com sucesso! Zero processamento extra.")
else:
    print(f"❌ ERRO: Arquivo '{arquivo_quantizado}' não encontrado na pasta atual!")
    exit()
print("==================================================\n")

# Verifica se o SHIFT e a Escala voltaram certinho
conv1_scale = model_int8.conv1.scale
conv1_shift = int(np.round(-np.log2(conv1_scale)))
print(f"Camada Conv1 - Escala: {conv1_scale:.6f} -> SHIFT Restaurado: {conv1_shift}")

# =====================================================================
# 4. FUNÇÃO DE EXTRAÇÃO E ALINHAMENTO PRO VHDL
# =====================================================================
def generate_vhdl_params_aligned(layer_name, layer_obj):
    bias_float = layer_obj.bias().detach().numpy()
    scale = layer_obj.scale
    if hasattr(scale, 'item'):
        scale = scale.item()
    
    bias_int8 = np.round(bias_float / scale).astype(np.int8)
    
    print(f"--- Relatório {layer_name} ---")
    print(f"Bias convertidos (Signed 8-bit): {bias_int8}")

    # Montagem da Linha 0 (BIAS)
    bias_line = np.zeros(16, dtype=np.int8)
    bias_line[:len(bias_int8)] = bias_int8
    all_rows = [bias_line]

    # Extração e alinhamento dos pesos por filtro
    weights_nd = layer_obj.weight().int_repr().numpy()
    num_filters = weights_nd.shape[0]

    for f in range(num_filters):
        filter_weights = weights_nd[f].flatten()
        for i in range(0, len(filter_weights), 16):
            chunk = filter_weights[i:i+16]
            padded_chunk = np.zeros(16, dtype=np.int8)
            padded_chunk[:len(chunk)] = chunk 
            all_rows.append(padded_chunk)

    # Escrita do Arquivo
    filename = f"{layer_name}_params.txt"
    with open(filename, "w") as file:
        for row in all_rows:
            hex_str = "".join(f"{b & 0xFF:02X}" for b in reversed(row))
            file.write(f"0x{hex_str}\n")
            
    print(f"✓ Arquivo '{filename}' gerado com {len(all_rows)} linhas.")
    print(f"✓ Alinhamento perfeito: Cada filtro tem o seu próprio padding de zeros!")

# Executa a mágica para a primeira camada
generate_vhdl_params_aligned("conv1", model_int8.conv1)

Carregando pesos e escalas de 'lenet5_quantized.pth'...
✓ Modelo INT8 restaurado com sucesso! Zero processamento extra.

Camada Conv1 - Escala: 0.069699 -> SHIFT Restaurado: 4
--- Relatório conv1 ---
Bias convertidos (Signed 8-bit): [-4  3  3  3  0 -4]
✓ Arquivo 'conv1_params.txt' gerado com 13 linhas.
✓ Alinhamento perfeito: Cada filtro tem o seu próprio padding de zeros!


C:\Users\Norian\AppData\Local\Temp\ipykernel_24772\4010658917.py:52: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = quant.prepare(model)
C:\Users\Norian\AppData\Local\Temp\ipykernel_24772\4010658917.py:53: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.qu

In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torch.ao.quantization as quant
from torch.ao.quantization import QuantStub, DeQuantStub
import numpy as np
import os

print("=== 1. PREPARANDO O AMBIENTE E DADOS ===")

# Baixa/Carrega o MNIST rapidinho só para pegar a imagem de teste
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)

# A Planta da Rede (Obrigatório para o PyTorch saber onde colocar os pesos)
class LeNet5_Quantized(nn.Module):
    def __init__(self, activation_type="relu"):
        super(LeNet5_Quantized, self).__init__()
        self.quant = QuantStub()
        self.dequant = DeQuantStub()
        self.activation = nn.ReLU()
        
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.quant(x)
        x = self.pool1(self.activation(self.conv1(x)))
        x = self.pool2(self.activation(self.conv2(x)))
        x = x.reshape(-1, 16 * 5 * 5)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)
        x = self.dequant(x)
        return x

print("=== 2. CARREGANDO O MODELO QUANTIZADO SALVO ===")

model = LeNet5_Quantized(activation_type="relu")
model.eval()
model.qconfig = quant.get_default_qconfig('fbgemm')
model_prepared = quant.prepare(model)
model_int8 = quant.convert(model_prepared)

arquivo_quantizado = "lenet5_quantized.pth"
if os.path.exists(arquivo_quantizado):
    model_int8.load_state_dict(torch.load(arquivo_quantizado))
    print(f"✓ Pesos de '{arquivo_quantizado}' carregados com sucesso!")
else:
    print(f"❌ ERRO: O arquivo '{arquivo_quantizado}' não está na pasta!")
    exit()

print("\n=== 3. GERANDO GABARITO E ARQUIVO TXT ===")

# Pega a primeira imagem do lote
dataiter = iter(train_loader)
images, labels = next(dataiter)
img_tensor = images[0] # Shape: [1, 32, 32]

# Adiciona a dimensão do Batch. Shape vira: [1, 1, 32, 32]
imagem_teste = img_tensor.unsqueeze(0)

# Passa pelo quantizador (Simula a conversão que o hardware faz)
entrada_quantizada = model_int8.quant(imagem_teste)

# Roda APENAS a Conv1
saida_conv1 = model_int8.conv1(entrada_quantizada)

# Extrai os inteiros puros para comparar com a BRAM
saida_inteira = saida_conv1.int_repr().numpy()

print(f"\n--- GABARITO OFICIAL: IMAGEM LABEL [{labels[0]}] ---")
print("As primeiras 6 posições da imagem de saída (Filtro 0, Linha 0):")
print(saida_inteira[0, 0, 0, :6])

# Gera o arquivo TXT para a FPGA
img_int8 = (img_tensor.numpy() * 127).astype(np.int8).flatten()

with open("input_image.txt", "w") as f:
    for i in range(0, len(img_int8), 16):
        chunk = img_int8[i:i+16]
        hex_str = "".join(f"{b & 0xFF:02X}" for b in reversed(chunk))
        f.write(f"0x{hex_str}\n")

print(f"\n✓ Imagem de teste gerada em 'input_image.txt'")

=== 1. PREPARANDO O AMBIENTE E DADOS ===
=== 2. CARREGANDO O MODELO QUANTIZADO SALVO ===
✓ Pesos de 'lenet5_quantized.pth' carregados com sucesso!

=== 3. GERANDO GABARITO E ARQUIVO TXT ===

--- GABARITO OFICIAL: IMAGEM LABEL [8] ---
As primeiras 6 posições da imagem de saída (Filtro 0, Linha 0):
[63 63 63 63 63 63]

✓ Imagem de teste gerada em 'input_image.txt'


C:\Users\Norian\AppData\Local\Temp\ipykernel_24772\1322406947.py:53: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = quant.prepare(model)
c:\Users\Norian\miniconda3\envs\ia_env\lib\site-packages\torch\ao\quantization\observer.py:246: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be

In [4]:
import numpy as np

print("\n=== 5. CALCULANDO O SHIFT REAL PARA O VHDL ===")

# 1. Escala da Imagem de Entrada (O QuantStub)
s_in = model_int8.quant.scale
if hasattr(s_in, 'item'): s_in = s_in.item()

# 2. Escala dos Pesos (O fbgemm geralmente usa uma escala por filtro!)
# Pegamos a escala do Filtro 0 para esse teste
s_weight = model_int8.conv1.weight().q_per_channel_scales()[0]
if hasattr(s_weight, 'item'): s_weight = s_weight.item()

# 3. Escala de Saída da Camada
s_out = model_int8.conv1.scale
if hasattr(s_out, 'item'): s_out = s_out.item()

# 4. A Fórmula Mágica da Quantização (O Fator M)
M = (s_in * s_weight) / s_out

print(f"Escala da Entrada: {s_in:.6f}")
print(f"Escala do Peso (Filtro 0): {s_weight:.6f}")
print(f"Escala da Saída: {s_out:.6f}")
print(f"Multiplicador Exato (M): {M:.6f}")

# 5. Convertendo M para SHIFT (M ≈ 1 / 2^SHIFT)
# Então: 2^SHIFT ≈ 1 / M  =>  SHIFT ≈ log2(1 / M)
if M > 0:
    real_shift = int(np.round(np.log2(1 / M)))
    print(f"\n=> O SHIFT REAL PARA O SEU VHDL É: {real_shift} <=")
else:
    print("\nErro: M deu zero ou negativo, tem algo errado nas escalas.")


=== 5. CALCULANDO O SHIFT REAL PARA O VHDL ===
Escala da Entrada: 0.015740
Escala do Peso (Filtro 0): 0.002223
Escala da Saída: 0.069699
Multiplicador Exato (M): 0.000502

=> O SHIFT REAL PARA O SEU VHDL É: 11 <=
